# Análisis de Operaciones — Alojamientos Turísticos
### StaySpain · Departamento de Analistas de Datos · Perfil: **Operaciones y Gestión de Inventario**

**Contexto del negocio.** StaySpain es una plataforma digital que conecta anfitriones y huéspedes en toda España. Como analistas del Departamento de Datos, nuestro trabajo es transformar la información de los anuncios en conocimiento estratégico para la dirección. En este informe analizamos la **disponibilidad del inventario** publicado en la plataforma desde la perspectiva de operaciones.

**Naturaleza de los datos.** El dataset es una *fotografía* (no una serie temporal): cada fila es un anuncio en un momento dado de extracción (`insert_date`). La **disponibilidad** (`availability_30/60/90/365`) la fija el anfitrión al gestionar su calendario, e indica los **días libres** en cada horizonte temporal. Conviene recordar que un día "no disponible" puede deberse tanto a una reserva real como a un bloqueo voluntario del calendario.

**Pregunta de negocio a responder.** ¿Cuál es la disponibilidad media de los alojamientos turísticos en los diferentes horizontes temporales (30, 60, 90 y 365 días) en cada ciudad?

**Datos limpios y deduplicados.** Trabajamos sobre 6.733 alojamientos únicos (una fila por anuncio, conservando la extracción más reciente).

> **Nota de alcance.** Este informe responde **únicamente a la pregunta del perfil de Operaciones**. La pregunta del perfil de Marketing (precio medio por tipo de alojamiento y ciudad) se aborda en un cuaderno independiente. Aquí no se analiza precio.


## 00 · Librerías

In [ ]:
#%pip install --upgrade nbformat
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

%matplotlib inline
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (9, 5)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\vanem\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 01 · Cargar dataset limpio

In [2]:
df_limpio = pd.read_parquet("2026_05_25_pisos_turisticos_limpio.parquet")
df_limpio.head()

,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,beds,amenities_list,price,minimum_nights,maximum_nights,has_availability,availability_30,availability_60,availability_90,availability_365,number_of_reviews,first_review_date,last_review_date,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date
0,11964,A ROOM WITH A VIEW,Private bedroom in our attic apartment. Right ...,45553,Centro,None,Private room,2,2.0,2.0,1.0,"TV,Internet,Wifi,Air conditioning,Elevator,Buz...",400.0,3,365,True,7,20,40,130,78,2010-01-02,2010-01-02,97.0,100.0,100.0,100.0,100.0,100.0,100.0,False,75.0,spain,malaga,2018-07-31
1,21853,Bright and airy room,We have a quiet and sunny room with a good vie...,83531,C�rmenes,Latina,Private room,1,1.0,1.0,1.0,"TV,Internet,Wifi,Air conditioning,Kitchen,Free...",170.0,4,40,True,0,0,0,162,33,2014-10-10,2014-10-10,92.0,90.0,90.0,100.0,100.0,80.0,90.0,False,52.0,spain,madrid,2020-01-10
2,32347,Explore Cultural Sights from a Family-Friendly...,Open French doors and step onto a plant-filled...,139939,San Vicente,Casco Antiguo,Entire home/apt,4,1.0,1.0,2.0,"TV,Internet,Wifi,Air conditioning,Wheelchair a...",990.0,2,120,True,26,31,31,270,148,2011-01-05,2011-01-05,98.0,100.0,100.0,100.0,100.0,100.0,100.0,True,142.0,spain,sevilla,2019-07-29
3,35379,Double 02 CasanovaRooms Barcelona,Room at a my apartment. Kitchen and 2 bathroom...,152232,l'Antiga Esquerra de l'Eixample,Eixample,Private room,2,2.0,2.0,1.0,"TV,Internet,Wifi,Kitchen,Breakfast,Elevator,Bu...",400.0,2,730,True,9,23,49,300,292,2012-03-13,2012-03-13,94.0,100.0,90.0,100.0,100.0,100.0,90.0,True,306.0,spain,barcelona,2020-01-10
4,35801,Can Torras Farmhouse Studio Suite,Lay in bed & watch sunlight change the mood of...,153805,Quart,None,Private room,5,1.0,1.0,5.0,"Wifi,Pool,Free parking on premises,Breakfast,P...",900.0,1,180,True,0,19,49,312,36,2011-07-08,2011-07-08,97.0,100.0,100.0,100.0,100.0,100.0,100.0,False,39.0,spain,girona,2019-02-19


**Lectura ejecutiva.** El dataset limpio contiene **6.733 alojamientos únicos** distribuidos por 8 ciudades españolas. La cartera está concentrada principalmente en Barcelona (2.041), Madrid (1.396), Mallorca (1.096) y Girona (1.085), mientras que destinos como Menorca (138), Valencia (297), Málaga (339) y Sevilla (341) son mucho más pequeños. Este desequilibrio en el tamaño de la muestra es importante: las medias serán más estables en las ciudades grandes y más ruidosas en las pequeñas.

## 02 · Filtrado de variables relevantes

In [3]:
df_limpio.columns

Index(['apartment_id', 'name', 'description', 'host_id', 'neighbourhood_name', 'neighbourhood_district', 'room_type', 'accommodates', 'bathrooms', 'bedrooms',
       'beds', 'amenities_list', 'price', 'minimum_nights', 'maximum_nights', 'has_availability', 'availability_30', 'availability_60', 'availability_90',
       'availability_365', 'number_of_reviews', 'first_review_date', 'last_review_date', 'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value',
       'is_instant_bookable', 'reviews_per_month', 'country', 'city', 'insert_date'],
      dtype='object')

In [4]:
# Variables que necesitamos para responder la pregunta de negocio
columnas = [
    "apartment_id",
    "room_type",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "city"
]

df_operaciones = df_limpio[columnas].copy()
df_operaciones.head()

,apartment_id,room_type,availability_30,availability_60,availability_90,availability_365,city
0,11964,Private room,7,20,40,130,malaga
1,21853,Private room,0,0,0,162,madrid
2,32347,Entire home/apt,26,31,31,270,sevilla
3,35379,Private room,9,23,49,300,barcelona
4,35801,Private room,0,19,49,312,girona


De las 35 columnas originales, para este análisis trabajamos solo con las **7 columnas operativas**: identificador, ciudad, tipo de habitación y las cuatro ventanas de disponibilidad. El resto de información (precio, reseñas, descripción…) queda fuera del foco de operaciones e inventario y la analizan otros perfiles del equipo.

## 03 · Distribución de la disponibilidad (histogramas)

Antes de calcular medias, conviene comprobar **cómo se distribuyen** los días disponibles en cada horizonte. Esto nos dirá si la media es una métrica fiable o si conviene usar la mediana.

In [5]:
variables = ["availability_30", "availability_60", "availability_90", "availability_365"]

fig = make_subplots(rows=2, cols=2, subplot_titles=variables)

row, col = 1, 1
for var in variables:
    fig.add_trace(
        go.Histogram(
            x=df_operaciones[var],
            nbinsx=40,
            marker_color='#2E74B5',
            opacity=0.8,
            name=var,
            showlegend=False
        ),
        row=row, col=col
    )
    col += 1
    if col == 3:
        col = 1
        row += 1

fig.update_layout(
    title='<b>Distribución de Disponibilidad</b>',
    title_x=0.5,
    width=1000,
    height=700,
    template='plotly_white'
)
fig.show()

**Interpretación operativa.** Las cuatro distribuciones **no son normales**: están claramente **sesgadas y con acumulaciones en los extremos**. Hay un patrón muy informativo para nuestro perfil:

- En **`availability_30`** se aprecian dos picos: muchos alojamientos con **0 días libres** (calendario completamente cerrado o reservado) y otro grupo con disponibilidad casi total. Es decir, conviven anuncios *muy ocupados* y anuncios *vacíos* sin apenas valores intermedios.
- A medida que ampliamos el horizonte (60 → 90 → 365), la distribución se hace más amplia: cuanto más a futuro miramos, menos restringidos están los calendarios.
- En **`availability_365`** la distribución se acerca a una uniforme amplia con un pico fuerte cerca de los 365 días (anuncios con todo el año disponible: probablemente *inventario inactivo* o de baja rotación).

**Implicaciones para el análisis.** Por la asimetría, **conviene reportar la mediana junto a la media** y leer los resultados con cautela en los extremos. Para tomar decisiones operativas a corto plazo (30 días) hay que segmentar: la media esconde dos comportamientos opuestos (anuncios saturados vs. anuncios libres).

## 03.1 · Outliers y dispersión (boxplots)

In [6]:
variables = ['availability_30', 'availability_60', 'availability_90', 'availability_365']
colores = ['#2E74B5', '#FF5733', '#28A745', '#8E44AD']

fig = go.Figure()

for i, var in enumerate(variables):
    fig.add_trace(go.Box(
        y=df_operaciones[var],
        name=var,
        boxpoints='outliers',
        marker_color=colores[i]
    ))

fig.update_layout(
    title='<b>Dispersión de la disponibilidad por horizonte</b>',
    title_x=0.5,
    yaxis_title='Días disponibles',
    width=1200,
    height=500,
    template='plotly_white',
    showlegend=False
)
fig.show()

**Interpretación operativa.** Los boxplots confirman la asimetría detectada en los histogramas:

- **`availability_30`** tiene una mediana de **alrededor de 10 días** y un rango intercuartílico estrecho (la mayoría de alojamientos están entre 0 y 20 días libres del próximo mes). Los outliers son anuncios con disponibilidad completa (30 días libres), que en este horizonte son sospechosos de ser inventario *inactivo*.
- **`availability_60`** y **`availability_90`** muestran un crecimiento ordenado de la mediana (~26 y ~46 días), sin grandes valores atípicos.
- **`availability_365`** presenta la mayor dispersión, con la mediana en torno a **180–200 días**. Es esperable: a un año vista el calendario aún no está cerrado.

**Conclusión metodológica.** Como las distribuciones son asimétricas, **complementaremos la media con la mediana** en los análisis por ciudad. La media puede estar inflada por los anuncios con todo el calendario libre.

## 04 · Disponibilidad media por ciudad

### 04.1 · Tabla de disponibilidad media por ciudad y horizonte

In [7]:
disp_final = df_operaciones.groupby('city')[['availability_30','availability_60',
                                  'availability_90','availability_365']].mean().round(2)
disp_final

,availability_30,availability_60,availability_90,availability_365
city,,,,
barcelona,11.11,25.63,42.50,182.09
girona,14.59,31.27,48.35,195.07
madrid,10.41,24.12,39.95,163.56
malaga,12.14,28.43,47.13,202.65
mallorca,13.41,28.70,45.17,210.92
menorca,14.99,30.88,47.54,199.49
sevilla,14.06,30.92,50.05,199.75
valencia,13.59,29.73,47.72,183.75


**Lectura de los números.** Esta tabla responde directamente a la pregunta de negocio del perfil de operaciones. Los valores son **días libres** en cada horizonte:

- En **`availability_30`** las medias van de **10,4 (Madrid)** a **15,0 (Menorca)**: una diferencia de ~5 días entre el destino más tenso y el más holgado.
- En **`availability_365`** las medias se mueven entre **163,6 (Madrid)** y **210,9 (Mallorca)**: una diferencia de ~47 días, mucho mayor en términos absolutos pero proporcionalmente similar.

**Insight clave.** **Madrid y Barcelona son los mercados más tensos** (menos disponibilidad libre = mayor ocupación estimada). Las **islas y destinos costeros** muestran más disponibilidad estructural. Más detalle visual en los gráficos siguientes.

### 04.2 · Visualización: barras agrupadas

In [8]:
fig = go.Figure()
for col in disp_final.columns:
    fig.add_trace(go.Bar(
        x=disp_final.index,
        y=disp_final[col],
        name=col
    ))

fig.update_layout(
    title='<b>Disponibilidad media por ciudad y horizonte</b>',
    title_x=0.5,
    xaxis_title='Ciudad',
    yaxis_title='Días disponibles (media)',
    width=1300, height=500,
    template='plotly_white'
)
fig.show()

**Interpretación.** El gráfico de barras confirma visualmente lo que veíamos en la tabla:

- A **365 días** (barra más oscura) la diferencia entre ciudades es notable: **Mallorca (211 días)** y **Málaga (203)** quedan en lo alto; **Madrid (164)** muy por debajo. Esto sugiere que las islas y costa tienen una porción importante del calendario sin cerrar a un año vista, mientras que Madrid trabaja con horizontes más cortos y comprometidos.
- A **30 días** (barra más clara) la diferencia se comprime porque los calendarios están más cerrados en todos los destinos. Aun así, Madrid sigue siendo el mercado con menos holgura inmediata.
- El patrón es **monótono** dentro de cada ciudad: más horizonte → más días libres. Esto valida la coherencia de los datos (no hay anomalías).

### 04.3 · Visualización: heatmap

In [9]:
fig = go.Figure(data=go.Heatmap(
    z=disp_final.values,
    x=disp_final.columns,
    y=disp_final.index,
    colorscale='Blues',
    colorbar_title='Días',
    text=disp_final.round(1).values,
    texttemplate='%{text}'
))

fig.update_layout(
    title='<b>Disponibilidad media por ciudad y horizonte (heatmap)</b>',
    title_x=0.5,
    width=900, height=500,
    template='plotly_white'
)
fig.show()

**Interpretación.** El heatmap permite captar el patrón de un solo vistazo: las celdas **más claras (azul pálido)** son las ciudades más ocupadas (Madrid y Barcelona en todos los horizontes), las **más oscuras (azul intenso)** los destinos con más holgura (Mallorca, Málaga, Sevilla). La columna `availability_365` es la que más contraste tiene, mostrando que **las diferencias estructurales entre ciudades se amplifican a largo plazo**.

### 04.4 · Mediana de la disponibilidad por ciudad (comparación con la media)

Como las distribuciones son **asimétricas** (lo confirmamos en los histogramas y boxplots de §03), conviene complementar la media con la mediana. La mediana indica el valor central efectivo (la mitad de los alojamientos por encima, la mitad por debajo) y resiste mejor a los anuncios con calendarios completamente abiertos o cerrados.

In [10]:
# Mediana de la disponibilidad por ciudad y horizonte
disp_mediana = df_operaciones.groupby('city')[['availability_30','availability_60',
                                  'availability_90','availability_365']].median().round(1)
disp_mediana

,availability_30,availability_60,availability_90,availability_365
city,,,,
barcelona,8.0,22.0,44.0,180.0
girona,16.0,35.0,56.0,197.0
madrid,7.0,21.0,41.0,146.0
malaga,10.0,29.0,52.0,215.0
mallorca,12.0,28.0,44.0,219.0
menorca,19.0,34.5,52.0,179.0
sevilla,13.0,30.0,54.0,211.0
valencia,13.0,31.0,52.0,177.0


In [11]:
fig = go.Figure(data=go.Heatmap(
    z=disp_mediana.values,
    x=disp_mediana.columns,
    y=disp_mediana.index,
    colorscale='Oranges',
    colorbar_title='Días',
    text=disp_mediana.values,
    texttemplate='%{text}'
))

fig.update_layout(
    title='<b>Disponibilidad MEDIANA por ciudad y horizonte</b>',
    title_x=0.5,
    width=900, height=500,
    template='plotly_white'
)
fig.show()

**Lectura comparada media vs. mediana.** La comparación revela matices importantes:

- **Madrid a 30 días:** media 10,4 días, **mediana 7,0**. La mediana es un 33% más baja → la media está inflada por anuncios con calendarios abiertos. **La mitad del inventario madrileño tiene 7 o menos días libres del próximo mes** — un mercado aún más tenso de lo que sugería la media.
- **Barcelona a 30 días:** media 11,1, **mediana 8,0**. Mismo patrón → segunda ciudad más tensa, también más de lo que la media indicaba.
- **Mallorca y Sevilla:** medias y medianas más próximas → distribuciones más simétricas; las medias son más representativas en estos destinos.
- **Menorca a 30 días:** media 15,0, **mediana 19,0**. Aquí la mediana es **mayor** que la media: unos pocos anuncios muy reservados arrastran la media hacia abajo, pero el anuncio típico de Menorca tiene aún más holgura.

**Conclusión metodológica.** La mediana **refuerza el diagnóstico** de mercados tensos en Madrid y Barcelona — son aún más tensos de lo que parecía a primera vista. En el resto de destinos, media y mediana cuentan una historia parecida, lo que da confianza al análisis.

## 05 · Disponibilidad media global de la cartera

In [12]:
media_disp_glob = df_operaciones[['availability_30', 'availability_60', 'availability_90', 'availability_365']].mean().reset_index(name='Media_dispo_global_dias').round(2)
media_disp_glob

,index,Media_dispo_global_dias
0,availability_30,12.29
1,availability_60,27.42
2,availability_90,44.30
3,availability_365,187.39


In [13]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=media_disp_glob['index'],
    y=media_disp_glob['Media_dispo_global_dias'],
    mode='lines+markers+text',
    line=dict(color='#2E74B5', width=3),
    marker=dict(size=14, color='#FF5733'),
    text=media_disp_glob['Media_dispo_global_dias'],
    textposition='top center',
    showlegend=False
))

fig.update_layout(
    title='<b>Disponibilidad media global de la cartera por horizonte</b>',
    title_x=0.5,
    xaxis_title='Horizonte',
    yaxis_title='Días disponibles (media)',
    template='plotly_white',
    width=900, height=450
)
fig.show()

**Interpretación operativa.** Globalmente, la cartera de StaySpain presenta:

| Horizonte | Días libres (media) | Ocupación estimada |
|---|---|---|
| 30 días  | **12,3**  | ≈ 59 % |
| 60 días  | **27,4**  | ≈ 54 % |
| 90 días  | **44,3**  | ≈ 51 % |
| 365 días | **187,4** | ≈ 49 % |

Convertido a **ocupación estimada** (1 − disponibilidad/horizonte), la cartera ronda el **59 % a 30 días** y baja gradualmente hasta el **49 % a 365 días**. Este descenso es el comportamiento esperado: a corto plazo los calendarios están más cerrados (compromisos firmes); a largo plazo, los anfitriones aún no han bloqueado fechas.

**Lectura de negocio.** Si la dirección establece un objetivo de **70 % de ocupación a 30 días** como referencia de salud del inventario, la cartera está **11 puntos por debajo del objetivo**. Esa brecha es la oportunidad concreta para Operaciones.

## 06 · Disponibilidad por ciudad y tipo de habitación

Bajamos un nivel de granularidad: ¿se comportan igual los apartamentos enteros que las habitaciones privadas? Esto importa porque las decisiones de inventario y promoción se toman por **segmento**, no solo por ciudad.

In [14]:
horizontes = ["availability_30", "availability_60", "availability_90", "availability_365"]

fig = make_subplots(rows=2, cols=2, subplot_titles=horizontes)

row, col = 1, 1
for h in horizontes:
    tabla = df_operaciones.pivot_table(
        index="city",
        columns="room_type",
        values=h,
        aggfunc="mean"
    )

    fig.add_trace(
        go.Heatmap(
            z=tabla.values,
            x=tabla.columns,
            y=tabla.index,
            colorscale="Blues",
            showscale=True if (row == 1 and col == 1) else False,
            text=tabla.round(1).values,
            texttemplate='%{text}'
        ),
        row=row, col=col
    )

    col += 1
    if col == 3:
        col = 1
        row += 1

fig.update_layout(
    title="<b>Disponibilidad media por ciudad y tipo de habitación</b>",
    title_x=0.5,
    width=1100, height=900,
    template="plotly_white"
)
fig.show()

**Lectura general — el patrón se mantiene pero con matices.** Los cuatro heatmaps confirman que **a mayor horizonte mayor disponibilidad**, y que **Madrid y Barcelona siguen siendo los mercados más tensos** en todos los tipos. Pero al desagregar por tipo, aparecen matices muy útiles para inventario.

**Por horizonte:**

🟦 **A 30 días (corto plazo)**: La presión es máxima en **`Entire home/apt` en Madrid (10,9 días libres) y Barcelona (10,9)**. Las **habitaciones compartidas** (Shared room) muestran un comportamiento muy errático: 0 días en Sevilla, 19 días en Barcelona. Eso indica que es un segmento marginal donde un solo anuncio puede mover mucho la media.

🟦 **A 60 días**: El patrón se amplifica pero se mantiene. Las ciudades grandes siguen siendo las más tensas; las islas y la costa, las más holgadas.

🟦 **A 90 días (trimestre)**: Empieza a verse la **elasticidad por segmento**. En Madrid las `Private room` empiezan a soltar disponibilidad mientras los `Entire home/apt` siguen comprometidos.

🟦 **A 365 días (anual)**: Aquí explotan las diferencias. **Mallorca, Málaga, Sevilla y Girona** superan los 180–215 días libres anuales en los apartamentos enteros, mientras que **Madrid se queda en 164 días**. Es la imagen más clara del **mercado tenso vs. mercado estacional**.

**Por tipo de habitación:**

🏠 **Entire home/apt** — el segmento más voluminoso y el que mejor representa la presión real: muy disputado en Madrid/Barcelona, mucho más holgado en islas/costa.

🏨 **Hotel room** — siempre con valores **moderadamente altos y consistentes**. Probablemente refleja la estrategia comercial de los hoteles, que mantienen disponibilidad para captar reservas de última hora.

🚪 **Private room** — el segmento **más elástico**: en ciudades tensas (Madrid) libera disponibilidad a horizontes largos, sugiriendo que es la palanca de absorción de demanda cuando los apartamentos enteros se agotan.

🛏️ **Shared room** — **segmento marginal y muy volátil**: en algunas ciudades es prácticamente residual (Sevilla, Menorca). No tomaría decisiones operativas basadas solo en este tipo.

## 07 · Ranking de ciudades por disponibilidad

In [15]:
ranking_general = (
    df_operaciones
    .groupby("city")[["availability_30","availability_60","availability_90","availability_365"]]
    .mean()
    .mean(axis=1)
    .sort_values(ascending=False)
    .round(1)
    .reset_index(name="Media_total_dias")
)
ranking_general

,city,Media_total_dias
0,mallorca,74.5
1,sevilla,73.7
2,menorca,73.2
3,malaga,72.6
4,girona,72.3
5,valencia,68.7
6,barcelona,65.3
7,madrid,59.5


**Lectura del ranking.** Si promediamos los cuatro horizontes en un único indicador agregado:

1. **Mallorca, Málaga, Sevilla, Menorca, Girona** lideran el ranking de mayor disponibilidad media (mercados más holgados, posiblemente más estacionales).
2. **Madrid y Barcelona** quedan al final (mercados más tensos, oferta absorbida por la demanda continua).
3. **Valencia** queda en posición intermedia.

**Interpretación de negocio (operativa).** Este ranking confirma que el modelo de negocio de StaySpain convive con **dos perfiles de inventario muy distintos**:

- **Mercados urbanos de demanda continua** (Madrid, Barcelona): saturación alta y calendarios cerrados. Oportunidad para **ampliar la base de anfitriones** y trabajar la **gestión del calendario** para maximizar la rotación.
- **Mercados costeros/insulares de demanda estacional** (Mallorca, Málaga, Menorca…): mucha disponibilidad fuera de temporada alta. Oportunidad para **reactivar inventario inactivo** y mejorar la **gestión estacional del calendario** por parte de los anfitriones.


## 08 · Limitaciones del análisis

Antes de pasar a las conclusiones y propuestas, hay que ser transparentes sobre lo que estos datos **no** pueden decirnos:

1. **Disponibilidad ≠ ocupación real.** Un día marcado como "no disponible" puede ser una **reserva real** o un **bloqueo voluntario del anfitrión** (vacaciones, mantenimiento, uso personal). Lo que medimos es la *capacidad ofertada*, no las reservas efectivas. Para una métrica de ocupación real haría falta cruzar con los datos del módulo de Reservas de StaySpain.
2. **Foto estática.** El dataset es una extracción puntual. No permite analizar **estacionalidad** ni evolución temporal real, aunque las ventanas a 30/60/90/365 días nos dan una aproximación del horizonte de planificación.
3. **Tamaño de muestra desigual.** Menorca (138), Valencia (297) y Málaga (339) tienen pocos anuncios; sus medias son más sensibles a outliers que las de Barcelona (2.041) o Madrid (1.396).
4. **Segmento Shared room marginal.** Con tan pocos anuncios en algunas ciudades, las medias de este segmento son poco fiables.

## 09 · Respuestas metodológicas (perfil Operaciones e Inventario)

### ¿Por qué esta técnica de análisis y no otra?
Para responder *"¿cuál es la disponibilidad media por ciudad y horizonte?"* se ha utilizado **estadística descriptiva agrupada** (medias y medianas por `city` y por ventana de disponibilidad), apoyada en **EDA visual** (histogramas, boxplots, heatmaps).

- **Tipo de datos:** observacionales y transversales (una *foto*), con variables numéricas (días de disponibilidad) y categóricas (ciudad, tipo de habitación). No hay variable temporal continua: `insert_date` es la fecha de extracción, no una serie de ocupación a lo largo del tiempo.
- **Objetivo:** *describir y comparar* el estado del inventario entre destinos, no predecir ni inferir relaciones causales.
- **Ventajas en este contexto:** la estadística descriptiva agrupada es directa, interpretable por negocio y suficiente para responder la pregunta. Una técnica predictiva (regresión, clustering) sería excesiva con datos estáticos para este objetivo. **No se ha aplicado correlación (Pearson o Spearman)** porque la pregunta es descriptiva, no relacional, y las correlaciones entre los cuatro horizontes solo confirmarían lo obvio (todos los horizontes miden lo mismo en distinto plazo).

### ¿Qué supuestos tiene y cómo se verificó que se cumplían?
- **Independencia de las observaciones:** inicialmente se incumplía por los duplicados (el mismo alojamiento aparecía en distintas extracciones). Se verificó mediante conteo de `apartment_id` duplicados y se corrigió **deduplicando** (una fila por anuncio, conservando la extracción más reciente). Tras la limpieza, **6.733 alojamientos únicos**.
- **Normalidad y outliers:** verificada con histogramas y boxplots (§03). Las distribuciones **no son normales** (sesgadas y con acumulación en los extremos). Por ello se reporta **mediana** junto a la media (§04.4) y se interpreta con cautela.
- **Tamaño de muestra:** 6.733 registros es amplio para estimaciones descriptivas por ciudad, salvo en destinos pequeños (Menorca, Valencia, Málaga), donde las medias son menos estables.

### ¿Qué limitaciones tiene y cómo afectan a las conclusiones?
La principal: **disponibilidad ≠ ocupación real**, así que todas las conclusiones sobre "presión de demanda" deben leerse como **estimaciones**, no medidas exactas.

**Lo que sí permite concluir el análisis:**
- Qué ciudades tienen, en promedio, más o menos inventario libre **(ver tabla §04.1, heatmap §04.3 y ranking §07)**.
- Cómo se reparte la disponibilidad por horizonte temporal **(ver gráfico y tabla globales §05)**.
- Qué segmentos (tipos de habitación) son más tensos o más holgados **(ver heatmaps cruzados city × room_type §06)**.
- Dónde la media está inflada por anuncios con calendarios extremos **(ver comparación media vs. mediana §04.4)**.

**Lo que no permite concluir:**
- La ocupación real (reservas efectivas).
- La causalidad de las diferencias entre destinos.
- La evolución temporal o la estacionalidad.

**Cómo mejorar el análisis en el futuro:**
- Integrar la **tabla de reservas reales** de StaySpain para distinguir reservas de bloqueos voluntarios.
- Disponer de datos **longitudinales** (varias capturas del calendario en el tiempo) para estudiar estacionalidad y rotación del inventario.
- Cruzar con datos de **actividad del anfitrión** (frecuencia de actualización del calendario, tasa de respuesta) para identificar inventario gestionado activamente vs. inactivo.


## 10 · Conclusiones y propuestas de negocio para StaySpain

### Conclusiones principales

1. **La cartera muestra dos comportamientos diferenciados según destino.** Madrid y Barcelona presentan las medias más bajas de disponibilidad libre a 30 días (**10,4 y 11,1 días libres** respectivamente; ver tabla §04.1 y heatmap §04.3), mientras que Menorca, Girona y Sevilla superan los **14 días libres** en el mismo horizonte. A 365 días la diferencia se amplifica: **Madrid 163,6 días vs. Mallorca 210,9 días**. La comparación media vs. mediana (§04.4) refuerza el diagnóstico: la mitad del inventario de Madrid tiene **7 o menos días libres** del próximo mes.

2. **La ocupación estimada a corto plazo es del 59 % (≈ 41 % por debajo del calendario completo).** Calculada como (30 − availability_30) / 30 sobre el conjunto: la disponibilidad media global a 30 días es de **12,3 días libres** (ver tabla y gráfico §05). Si la dirección define un objetivo orientativo del 70 % de ocupación, la brecha es de **11 puntos porcentuales**.

3. **El segmento `Entire home/apt` concentra el volumen y refleja la presión del mercado; `Private room` es elástico en ciudades tensas.** En Madrid, los apartamentos enteros tienen **10,9 días libres** a 30 días mientras los `Private room` caen a **9,4** (ver heatmap §06, fila *madrid*). El `Shared room` queda fuera del análisis robusto por bajo volumen (ver limitaciones §08).

4. **Existe inventario potencialmente inactivo.** Los histogramas de §03 muestran un pico de anuncios con disponibilidad cercana al máximo en los cuatro horizontes (sobre todo a 365 días), patrón compatible con anuncios sin gestión activa que inflan la oferta sin convertir.

### Propuestas priorizadas para la Directora (foco: Operaciones e Inventario)

| # | Propuesta | Hallazgo que la justifica | Prioridad | Recursos necesarios |
|---|---|---|---|---|
| 1 | **Detección y desactivación de inventario inactivo** | Pico de anuncios con disponibilidad cercana al máximo en los cuatro horizontes, especialmente a 365 días (ver histogramas §03 y comparación media vs. mediana §04.4). Patrón compatible con inventario "fantasma" sin gestión activa. | **Alta** | Producto + Atención al anfitrión + IT (segmentación) |
| 2 | **Plan de captación de inventario en mercados tensos** | Madrid presenta la media más baja de disponibilidad a 30 días (**10,4 días libres**, mediana **7,0**) y Barcelona la segunda (**11,1**, mediana **8,0**) — ver heatmap §04.3 y ranking §07. La presión sostenida indica demanda no satisfecha. | **Alta** | Captación + Marketing (sin afectar al pricing) |
| 3 | **Reactivación operativa del inventario estacional** | Mallorca, Málaga, Sevilla y Girona superan los **180–215 días libres anuales** (§04.1, columna `availability_365`). Indica calendarios poco gestionados fuera de temporada alta. Acompañar a estos anfitriones con guías de gestión de calendario. | **Media** | Atención al anfitrión + Contenidos |
| 4 | **Recomendaciones de gestión por segmento** | El segmento `Private room` libera disponibilidad en horizontes largos en Madrid y Barcelona (ver heatmap §06, fila *madrid*). El `Shared room` muestra valores muy dispares entre ciudades (de 0 a 19 días a 30 días), reflejando bajo volumen y poca fiabilidad como base para decisiones. | **Media** | Producto + Contenidos |

### Próximos pasos

Para profundizar y dar más solidez al análisis operativo, se recomienda solicitar a IT/Sistemas el acceso a:

- **Tabla de reservas reales** (para distinguir ocupación efectiva de bloqueos voluntarios del anfitrión).
- **Histórico longitudinal** (varias capturas del calendario en el tiempo) para estudiar **estacionalidad** y **rotación**.
- **Métricas de actividad del anfitrión** (última edición del calendario, tasa de respuesta a solicitudes) para identificar **inventario gestionado activamente vs. inactivo**.

Con esos datos, el siguiente análisis podría incluir la **tasa de conversión del inventario** (anuncios publicados → reservados), la **estacionalidad por destino** y una segmentación del inventario por nivel de actividad del anfitrión.

---

*Elaborado por: Departamento de Analistas de Datos · Equip 29 · Perfil de Operaciones e Inventario*
